<div style="padding: 28px 32px; border-radius: 18px; background: linear-gradient(135deg, #0f172a 0%, #1d4ed8 45%, #38bdf8 100%); color: white; box-shadow: 0 20px 50px rgba(15, 23, 42, 0.18);">
  <p style="margin: 0; font-size: 14px; letter-spacing: 0.18em; text-transform: uppercase; opacity: 0.85;">Customer Intelligence • Revenue Forecasting System</p>
  <h1 style="margin: 12px 0 10px; font-size: 34px;">Exploratory Data Analysis</h1>
  <p style="margin: 0; font-size: 17px; max-width: 820px; line-height: 1.7;">
    This notebook converts the cleaned transaction table into an executive-ready performance story across revenue, profitability,
    geography, products, customer value, discount impact, and segment quality.
  </p>
</div>


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import HTML, display
from matplotlib.ticker import FuncFormatter

pd.options.display.float_format = "{:,.2f}".format
sns.set_theme(style="whitegrid", context="talk")

project_root = Path.cwd()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from scripts.data_utils import load_cleaned_data
from scripts.export_utils import build_executive_kpis

def currency(x, pos=None):
    return f"${x:,.0f}"

def render_kpis(metrics):
    cards = []
    for title, value, color in metrics:
        cards.append(
            f'''
            <div style="flex: 1; min-width: 210px; background: white; border: 1px solid #e2e8f0; border-top: 5px solid {color};
                        border-radius: 18px; padding: 18px 20px; box-shadow: 0 10px 24px rgba(15, 23, 42, 0.06);">
                <div style="font-size: 13px; text-transform: uppercase; letter-spacing: 0.08em; color: #64748b;">{title}</div>
                <div style="margin-top: 8px; font-size: 28px; font-weight: 700; color: #0f172a;">{value}</div>
            </div>
            '''
        )
    display(HTML('<div style="display:flex; gap:16px; flex-wrap:wrap;">' + "".join(cards) + '</div>'))

df = load_cleaned_data(project_root)
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])
df["year_month"] = pd.to_datetime(df["year_month"])


## Executive KPIs


In [ ]:
kpis = build_executive_kpis(df)
render_kpis(
    [
        ("Total Sales", f"${kpis.loc[0, 'total_sales']:,.0f}", "#2563eb"),
        ("Total Profit", f"${kpis.loc[0, 'total_profit']:,.0f}", "#16a34a"),
        ("Total Orders", f"{int(kpis.loc[0, 'total_orders']):,}", "#06b6d4"),
        ("Total Customers", f"{int(kpis.loc[0, 'total_customers']):,}", "#f59e0b"),
        ("Avg Order Value", f"${kpis.loc[0, 'avg_order_value']:,.2f}", "#7c3aed"),
        ("Avg Sales / Customer", f"${kpis.loc[0, 'avg_sales_per_customer']:,.2f}", "#0f766e"),
    ]
)
kpis


## Time Analysis


In [ ]:
monthly = df.groupby("year_month")[["sales", "profit"]].sum().reset_index()
yearly = df.groupby("order_year")[["sales", "profit"]].sum().reset_index()

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

axes[0, 0].plot(monthly["year_month"], monthly["sales"], color="#2563eb", linewidth=2.5)
axes[0, 0].set_title("Monthly Sales Trend")
axes[0, 0].yaxis.set_major_formatter(FuncFormatter(currency))

axes[0, 1].plot(monthly["year_month"], monthly["profit"], color="#dc2626", linewidth=2.5)
axes[0, 1].axhline(0, linestyle="--", color="#334155")
axes[0, 1].set_title("Monthly Profit Trend")
axes[0, 1].yaxis.set_major_formatter(FuncFormatter(currency))

sns.barplot(data=yearly, x="order_year", y="sales", hue="order_year", legend=False, palette="Blues", ax=axes[1, 0])
axes[1, 0].set_title("Yearly Sales Comparison")
axes[1, 0].yaxis.set_major_formatter(FuncFormatter(currency))

seasonality = df.groupby("month_name")["sales"].sum().reindex(
    ["January","February","March","April","May","June","July","August","September","October","November","December"]
)
axes[1, 1].bar(seasonality.index, seasonality.values, color="#0ea5e9")
axes[1, 1].set_title("Seasonality by Month")
axes[1, 1].tick_params(axis="x", rotation=45)
axes[1, 1].yaxis.set_major_formatter(FuncFormatter(currency))

plt.tight_layout()
plt.show()


> **Insight:** Time trends reveal both growth and volatility. Seasonality matters because retail planning, staffing, and inventory decisions should align with predictable peaks and troughs.


## Geographic Analysis


In [ ]:
country_sales = df.groupby("country")["sales"].sum().sort_values(ascending=False).head(10)
region_profit = df.groupby("region")["profit"].sum().sort_values()
market_sales = df.groupby("market")["sales"].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
axes[0].barh(country_sales.index, country_sales.values, color="#1d4ed8")
axes[0].invert_yaxis()
axes[0].set_title("Top Countries by Sales")
axes[0].xaxis.set_major_formatter(FuncFormatter(currency))

axes[1].bar(region_profit.index, region_profit.values, color=["#dc2626" if x < 0 else "#16a34a" for x in region_profit.values])
axes[1].set_title("Profit by Region")
axes[1].tick_params(axis="x", rotation=25)
axes[1].yaxis.set_major_formatter(FuncFormatter(currency))

axes[2].bar(market_sales.index, market_sales.values, color="#06b6d4")
axes[2].set_title("Sales by Market")
axes[2].tick_params(axis="x", rotation=25)
axes[2].yaxis.set_major_formatter(FuncFormatter(currency))

plt.tight_layout()
plt.show()


> **Insight:** Strong markets should be judged on both scale and quality. A region can lead in sales while underperforming in profit, which usually signals pricing or cost pressure.


## Product Analysis


In [ ]:
category_perf = df.groupby("category")[["sales", "profit"]].sum().reset_index()
subcategory_profit = df.groupby("sub_category")["profit"].sum().sort_values().reset_index()
top_products = df.groupby("product_name")["sales"].sum().sort_values(ascending=False).head(10).reset_index()
loss_products = df.groupby("product_name")["profit"].sum().sort_values().head(10).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
sns.barplot(data=category_perf, x="category", y="sales", hue="category", legend=False, palette="crest", ax=axes[0, 0])
axes[0, 0].set_title("Sales by Category")
axes[0, 0].yaxis.set_major_formatter(FuncFormatter(currency))

sns.barplot(data=category_perf, x="category", y="profit", hue="category", legend=False, palette="flare", ax=axes[0, 1])
axes[0, 1].set_title("Profit by Category")
axes[0, 1].yaxis.set_major_formatter(FuncFormatter(currency))

axes[1, 0].barh(subcategory_profit["sub_category"], subcategory_profit["profit"], color=["#dc2626" if x < 0 else "#2563eb" for x in subcategory_profit["profit"]])
axes[1, 0].set_title("Profit by Sub-Category")
axes[1, 0].xaxis.set_major_formatter(FuncFormatter(currency))

axes[1, 1].barh(top_products["product_name"], top_products["sales"], color="#0f766e")
axes[1, 1].invert_yaxis()
axes[1, 1].set_title("Top Products by Sales")
axes[1, 1].xaxis.set_major_formatter(FuncFormatter(currency))

plt.tight_layout()
plt.show()

loss_products


> **Insight:** Product mix analysis separates healthy volume from expensive volume. That distinction matters for pricing strategy, inventory focus, and margin improvement.


## Customer Analysis


In [ ]:
top_sales_customers = df.groupby("customer_name")["sales"].sum().sort_values(ascending=False).head(10)
top_profit_customers = df.groupby("customer_name")["profit"].sum().sort_values(ascending=False).head(10)
orders_per_customer = df.groupby("customer_id")["order_id"].nunique()

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
axes[0].barh(top_sales_customers.index, top_sales_customers.values, color="#2563eb")
axes[0].invert_yaxis()
axes[0].set_title("Top Customers by Sales")
axes[0].xaxis.set_major_formatter(FuncFormatter(currency))

axes[1].barh(top_profit_customers.index, top_profit_customers.values, color="#16a34a")
axes[1].invert_yaxis()
axes[1].set_title("Top Customers by Profit")
axes[1].xaxis.set_major_formatter(FuncFormatter(currency))

sns.histplot(orders_per_customer, bins=30, kde=True, color="#06b6d4", ax=axes[2])
axes[2].set_title("Order Frequency Distribution")
axes[2].set_xlabel("Unique Orders per Customer")

plt.tight_layout()
plt.show()

repeat_share = (orders_per_customer.gt(1).mean() * 100)
print(f"Repeat buyer share: {repeat_share:.1f}%")


> **Insight:** Customer concentration and repeat buying behavior reveal whether growth is broad-based or dependent on a narrower set of relationships.


## Discount and Profitability Analysis


In [ ]:
profit_margin_by_category = df.groupby("category")["profit_margin"].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.scatterplot(data=df.sample(min(len(df), 10000), random_state=42), x="discount", y="profit", alpha=0.35, color="#0f766e", ax=axes[0])
axes[0].set_title("Discount vs Profit")

sns.barplot(data=profit_margin_by_category, x="category", y="profit_margin", hue="category", legend=False, palette="viridis", ax=axes[1])
axes[1].set_title("Average Profit Margin by Category")
axes[1].yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")

plt.tight_layout()
plt.show()


> **Insight:** Discounts can support volume, but if profit drops too sharply they become a growth illusion rather than a healthy revenue driver.


## Segment Analysis


In [ ]:
segment_perf = df.groupby("segment")[["sales", "profit"]].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=segment_perf, x="segment", y="sales", hue="segment", legend=False, palette="Blues", ax=axes[0])
axes[0].set_title("Sales by Customer Segment")
axes[0].yaxis.set_major_formatter(FuncFormatter(currency))

sns.barplot(data=segment_perf, x="segment", y="profit", hue="segment", legend=False, palette="Greens", ax=axes[1])
axes[1].set_title("Profit by Customer Segment")
axes[1].yaxis.set_major_formatter(FuncFormatter(currency))

plt.tight_layout()
plt.show()

segment_perf.assign(profit_margin=segment_perf["profit"] / segment_perf["sales"]).sort_values("profit", ascending=False)


## Save Dashboard-Ready KPI Output


In [ ]:
processed_dir = project_root / "data" / "processed"
outputs_dir = project_root / "outputs" / "csv"
processed_dir.mkdir(parents=True, exist_ok=True)
outputs_dir.mkdir(parents=True, exist_ok=True)

kpis.to_csv(processed_dir / "executive_kpis.csv", index=False)
df.to_csv(processed_dir / "cleaned_orders.csv", index=False)

print("Saved executive KPI table and cleaned orders export.")
